# Análisis de Datos · Semana 16, sesión 1 de 2
## Visualización y matplotlib

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

Una gráfica es un argumento. Todo lo de esta sesión existe para que ese argumento lo entienda
alguien que no estuvo en el cuarto cuando lo hiciste.

Al terminar este cuaderno vas a poder:

1. Elegir la gráfica por la pregunta: barra, línea, dispersión o histograma.
2. Construir una gráfica con `matplotlib`, con figura y ejes, y guardarla como imagen.
3. Titular con el hallazgo y no con los nombres de los ejes.
4. Formatear los ejes para que nadie tenga que contar dígitos.
5. Elegir color accesible, y no dejar que el color sea la única señal.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden. Las gráficas aparecen debajo de la celda que las dibuja, así que
vas a ver el efecto de cada cambio inmediatamente.

Dos celdas dibujan a propósito una gráfica mala, para que compares. Llevan un comentario que
lo dice.

---
## Preparación

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)

In [ ]:
# Plomería, no lección. Esta celda deja los tres CSV del curso al
# alcance de pandas y no vuelve a hacer falta.
#
# Primero los busca en el repositorio, que es público y se lee por URL.
# Si no responde, los reconstruye aquí mismo con la semilla fija del
# curso, así que salen idénticos por cualquiera de los dos caminos.
# En ningún caso hay que subir un archivo a mano.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
ARCHIVOS = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in ARCHIVOS:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Vuelve a escribir los tres CSV con la semilla fija del curso.

    Salen idénticos byte por byte a los del repositorio, así que los
    números de la diapositiva siguen coincidiendo con los del cuaderno.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # la suciedad deliberada: una región tecleada de cuatro formas,
    # celdas en blanco, y renglones capturados dos veces
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Datos leídos del repositorio.")
except Exception:
    _reconstruir_datos()
    print("El repositorio no respondió. Datos reconstruidos en esta sesión.")

print("Listos:", ", ".join(ARCHIVOS))

In [ ]:
# La limpieza de la sesión 15.2, en una celda, para que este cuaderno se abra solo.
ventas = pd.read_csv("sales.csv").drop_duplicates()
ventas["region"] = ventas["region"].str.strip().str.title()
ventas["unit_price"] = (ventas["unit_price"]
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.strip().astype(float))
ventas["date"] = pd.to_datetime(ventas["date"])
ventas = ventas.dropna(subset=["units"])
ventas["units"] = ventas["units"].astype(int)
ventas["amount"] = ventas["units"] * ventas["unit_price"]

empleados = pd.read_csv("employees.csv")
mensual = ventas.groupby(ventas["date"].dt.month)["amount"].sum()

MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

print(f"{len(ventas)} renglones limpios, {len(empleados)} empleados")

---
# Bloque 1 · Cuál gráfica

No es una decisión de estilo. Cada forma contesta una pregunta, y usar la equivocada hace que
un número cierto diga algo falso.

| Gráfica | La pregunta que contesta | Ejemplo del curso |
|---|---|---|
| Barra | ¿Cómo se comparan estas categorías? | Ingreso por producto |
| Línea | ¿Cómo cambió esto con el tiempo? | Ingreso por mes |
| Dispersión | ¿Estas dos cifras se mueven juntas? | Sueldo contra antigüedad |
| Histograma | ¿Cómo se reparten los valores? | Distribución de sueldos |

Las cuatro, dibujadas con los datos del curso, una por una.

## Barra: comparar categorías

Ordenada, porque una gráfica de barras sin ordenar obliga al lector a hacer el ranking a ojo.
Horizontal, porque los nombres de las categorías son palabras y las palabras se leen a lo
ancho.

In [ ]:
por_producto = ventas.groupby("product")["amount"].sum().sort_values() / 1000

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(por_producto.index, por_producto.values, color="#2B5F8F")
ax.set_title("¿Qué producto trae más ingreso?", loc="left", fontweight="bold")
ax.set_xlabel("Miles de pesos")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

La máquina de espresso trae más de la mitad del ingreso, y el orden de las barras ya contestó
la pregunta sin que nadie tenga que comparar longitudes.

## Línea: el cambio a lo largo de un eje ordenado

Una línea le dice al lector que los puntos están conectados en un orden que significa algo.
Eso es cierto entre enero y febrero.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(MESES, mensual.values / 1000, marker="o", linewidth=2, color="#2B5F8F")
ax.set_title("¿Cómo se movió el ingreso durante el año?", loc="left", fontweight="bold")
ax.set_ylabel("Miles de pesos")
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

### La regla que sigue de eso

Conectar dos puntos con una línea afirma que hay un recorrido entre ellos. Entre enero y
febrero es cierto. Entre Norte y Sur es falso, y el lector se lo va a creer porque la forma se
lo está diciendo.

In [ ]:
# DIBUJA MAL A PROPÓSITO. Una línea sobre categorías inventa una trayectoria.
por_region = ventas.groupby("region")["amount"].sum() / 1000

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(por_region.index, por_region.values, marker="o", color="#B4530A", linewidth=2)
axes[0].set_title("Mal: ¿Norte lleva a Centre?", loc="left", fontweight="bold")

axes[1].bar(por_region.index, por_region.values, color="#2B5F8F")
axes[1].set_title("Bien: cuatro cosas comparables", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Miles de pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

Las dos gráficas traen exactamente los mismos cuatro números. La de la izquierda sugiere que
las regiones están en una secuencia y que hay una caída de Centre a South, cuando el orden es
alfabético y no significa nada.

## Dispersión: la relación entre dos cifras

Un punto por renglón, colocado por dos de sus valores. Es la gráfica que contesta si más de
esto viene con más de aquello.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(empleados["tenure_months"], empleados["monthly_salary"],
           alpha=0.55, color="#2B5F8F", edgecolor="none")
ax.set_title("¿El sueldo sube con la antigüedad?", loc="left", fontweight="bold")
ax.set_xlabel("Antigüedad en meses")
ax.set_ylabel("Sueldo mensual")

r = empleados["tenure_months"].corr(empleados["monthly_salary"])
ax.annotate(f"correlación = {r:.2f}", xy=(0.04, 0.92), xycoords="axes fraction",
            fontsize=10, color="#5B6B84")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

print("Correlación:", round(r, 3))

La correlación le pone número a lo que el ojo está haciendo. Va de menos uno a uno.

Aquí sale 0.28, que es una relación débil: la nube sube un poco a la derecha y aun así hay
gente con dos años ganando más que gente con diez. Un número cerca de cero significa que la
nube no tiene dirección, y un número fuerte **sigue sin significar** que uno causó al otro.

## Histograma: cómo se reparte una columna

Un histograma rebana una columna en rangos y cuenta cuántos renglones caen en cada uno.
Contesta cómo se ve lo típico y qué tan ancho es el reparto.

Una gráfica de barras compara cosas con nombre; un histograma compara rangos de una sola
cosa. Es la diferencia que más se confunde de las cuatro.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(empleados["monthly_salary"], bins=15, color="#2B5F8F", edgecolor="white")
ax.set_title("¿Cómo se reparten los sueldos?", loc="left", fontweight="bold")
ax.set_xlabel("Sueldo mensual")
ax.set_ylabel("Empleados")

promedio = empleados["monthly_salary"].mean()
ax.axvline(promedio, color="#B4530A", linestyle="--", linewidth=2)
ax.annotate(f"promedio {promedio:,.0f}", xy=(promedio, 0), xytext=(6, 6),
            textcoords="offset points", color="#B4530A", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

print("Promedio:", round(promedio))
print("Mediana: ", empleados["monthly_salary"].median())

El promedio dibujado encima muestra cuánto esconde. La mayoría de la gente gana por debajo de
él, y unos cuantos sueldos altos lo jalan hacia la derecha. Reportar solo el promedio de esta
columna daría una idea equivocada de lo que gana una persona típica.

## La que casi nunca sirve

Un pastel pide comparar ángulos, que es algo que la gente hace mal. Pasando de tres rebanadas
deja de leerse. Dibuja las dos con los mismos números y la diferencia se ve sola.

In [ ]:
# DIBUJA MAL A PROPÓSITO, del lado izquierdo. Los dos paneles traen los mismos datos.
partes = ventas.groupby("product")["amount"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].pie(partes.values, labels=partes.index, autopct="%1.0f%%",
            startangle=90, colors=plt.cm.Blues(range(60, 260, 40)))
axes[0].set_title("En pastel: ¿cuáles dos se parecen más?", loc="left", fontweight="bold")

axes[1].barh(partes.sort_values().index, partes.sort_values().values / 1000, color="#2B5F8F")
axes[1].set_title("En barras: ahora sí se nota", loc="left", fontweight="bold")
axes[1].set_xlabel("Miles de pesos")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.tight_layout()
plt.show()

---
# Bloque 2 · Cómo se construye

Dos objetos, y toda gráfica de matplotlib empieza con la misma línea.

Una **figura** es la hoja de papel. Unos **ejes** son un par de ejes dibujados sobre ella.
`subplots()` te entrega las dos cosas de golpe, y así empieza prácticamente toda gráfica que
vas a escribir.

Se dibuja en los ejes, y se guarda la figura.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(MESES, mensual.values / 1000)

fig.savefig("primera.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Escrito primera.png")

`dpi` controla qué tan nítido sale el archivo: 150 alcanza para proyectar, 300 para imprimir.
`bbox_inches="tight"` recorta el margen blanco de sobra.

`plt.close(fig)` cierra la figura al terminar. Un ciclo que dibuja cincuenta y no las cierra
las deja las cincuenta en memoria, y matplotlib acaba avisándotelo.

## Lo que le falta a esa gráfica

La de arriba es técnicamente correcta y no dice nada. No tiene título, los números del eje no
están etiquetados, y el lector tiene que adivinar qué significa el 1 al 12.

| Elemento | Qué aporta | Método |
|---|---|---|
| Título | El hallazgo, en una frase | `set_title` |
| Etiqueta de eje | Qué se mide, y en qué unidad | `set_ylabel` |
| Base en cero | Que la diferencia no se exagere | `set_ylim` |
| Fuente | De dónde salieron los números | `fig.text` |

Los mismos datos, contados bien.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(MESES, mensual.values / 1000, marker="o", linewidth=2, color="#2B5F8F")

ax.set_title("Ingreso por mes, 2025", fontsize=14, fontweight="bold", loc="left")
ax.set_ylabel("Miles de pesos")
ax.set_ylim(bottom=0)          # una barra o una línea empiezan en cero, o mienten

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.show()
plt.close(fig)

Todo lo que se quitó del marco era tinta que no estaba diciendo nada.

## El eje cortado, que es como se miente con números ciertos

`set_ylim(bottom=0)` no es decoración. Cortar el eje exagera la diferencia, y hacerlo a
propósito es la forma más común de mentir con una gráfica que solo contiene números
correctos.

In [ ]:
# DIBUJA MAL A PROPÓSITO, del lado izquierdo. Los mismos cuatro números en los dos.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(por_region.index, por_region.values, color="#B4530A")
axes[0].set_ylim(1400, 4500)                      # el eje cortado
axes[0].set_title("Mal: South parece no existir", loc="left", fontweight="bold")

axes[1].bar(por_region.index, por_region.values, color="#2B5F8F")
axes[1].set_ylim(bottom=0)
axes[1].set_title("Bien: South vende un tercio de North", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Miles de pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("North contra South:", round(por_region["North"] / por_region["South"], 2), "veces")

North vende 2.8 veces lo de South. En la gráfica de la izquierda parece veinte veces. Los
cuatro números son los mismos y ninguno está mal.

## Varias gráficas a la vez

`subplots` acepta una cuadrícula. Los ejes regresan como un arreglo que se indexa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

por_region_ord = ventas.groupby("region")["amount"].sum().sort_values() / 1000
por_canal = ventas.groupby("channel")["amount"].sum().sort_values() / 1000

axes[0].barh(por_region_ord.index, por_region_ord.values, color="#3776AB")
axes[0].set_title("Por región", loc="left", fontweight="bold")

axes[1].barh(por_canal.index, por_canal.values, color="#3776AB")
axes[1].set_title("Por canal", loc="left", fontweight="bold")

for ax in axes:
    ax.set_xlabel("Miles de pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()          # evita que las etiquetas de una encimen a la otra
plt.show()
plt.close(fig)

---
# Bloque 3 · Que se entienda sin ti

La gráfica va a viajar sola en un correo. Todo lo que tengas que explicar en voz alta le falta
escrito.

## El título dice el hallazgo

"Ingreso por mes" describe los ejes, que el lector ya está viendo. "Diciembre concentró el
20 % del ingreso del año" es lo que de verdad encontraste.

Una gráfica titulada con su conclusión se lee una vez. Una titulada con sus ejes se queda
mirando hasta que alguien la explica.

In [ ]:
pico = mensual.idxmax()
parte = mensual.max() / mensual.sum()

print(f"El mes pico es el {pico} y se llevó {parte:.1%} del año")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

barras = ax.bar(MESES, mensual.values, color="#C7D6E8", edgecolor="none")

# Una sola barra carga el argumento, así que una sola lleva el color fuerte.
barras[pico - 1].set_color("#2B5F8F")

ax.set_title(f"Diciembre concentró el {parte:.0%} del ingreso del año",
             fontsize=15, fontweight="bold", loc="left", pad=18)

# El subtítulo es donde va la descripción, ahora que el título dice lo que importa.
ax.text(0, 1.02, "Ingreso por mes, 2025", transform=ax.transAxes,
        fontsize=10.5, color="#5B6B84")

# 2567118.5 obliga a contar dígitos. 2.6M se lee sin pensarlo.
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / 1_000_000:.1f}M"))
ax.set_ylabel("Ingreso")
ax.set_ylim(bottom=0)

ax.annotate(f"{mensual.max() / 1_000_000:.2f}M",
            xy=(pico - 1, mensual.max()), xytext=(0, 8), textcoords="offset points",
            ha="center", fontweight="bold", color="#2B5F8F")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="y", length=0)

# Una gráfica sin fuente es una opinión.
fig.text(0.125, -0.02, "Fuente: sales_clean.csv, 306 registros, 2025",
         fontsize=9, color="#5B6B84")

plt.show()
plt.close(fig)

Cuatro cosas cambiaron respecto a la versión anterior, y ninguna tocó los datos.

El **título** dice el hallazgo y el subtítulo se quedó con la descripción. Una **sola barra**
lleva el color intenso: si todo resalta, nada resalta, y las otras once siguen ahí y siguen
siendo comparables, solo dejaron de competir por la atención. El **formateador** cambia las
etiquetas del eje sin tocar los valores de abajo. Y la **fuente** al pie convierte una opinión
en evidencia.

## Color que sobrevive al gris

Alrededor de uno de cada doce hombres tiene alguna forma de daltonismo, y toda gráfica acaba
tarde o temprano impresa en blanco y negro. Dos defensas:

1. **Usa una paleta pensada para eso.** Azul contra naranja se separa para casi todo el
   mundo; rojo contra verde no.
2. **No dejes que el color sea la única señal.** El estilo de línea, la forma del marcador y
   una etiqueta directa sobreviven todos a volverse grises.

In [ ]:
por_canal_mes = ventas.pivot_table(index=ventas["date"].dt.month,
                                   columns="channel", values="amount", aggfunc="sum")

SEGURO = {"Retail": "#2B5F8F", "Online": "#B4530A", "Wholesale": "#5B6B84"}
ESTILO = {"Retail": "-", "Online": "--", "Wholesale": ":"}
MARCA = {"Retail": "o", "Online": "s", "Wholesale": "^"}

fig, ax = plt.subplots(figsize=(10, 5))

for canal in por_canal_mes.columns:
    ax.plot(MESES, por_canal_mes[canal] / 1000, label=canal, color=SEGURO[canal],
            linestyle=ESTILO[canal], marker=MARCA[canal], linewidth=2)

    # Una etiqueta al final de la línea le gana a una leyenda: el ojo nunca tiene
    # que salirse de los datos para averiguar cuál línea es cuál.
    ax.annotate(canal, xy=(11, por_canal_mes[canal].iloc[-1] / 1000),
                xytext=(8, 0), textcoords="offset points",
                color=SEGURO[canal], fontweight="bold", va="center")

ax.set_title("Mayoreo es lo que produce el pico de diciembre",
             fontsize=15, fontweight="bold", loc="left", pad=18)
ax.text(0, 1.02, "Ingreso por canal y mes, en miles de pesos",
        transform=ax.transAxes, fontsize=10.5, color="#5B6B84")
ax.set_ylim(bottom=0)
ax.set_xlim(-0.4, 12.6)          # espacio a la derecha para las etiquetas
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.show()
plt.close(fig)

Esa gráfica se sigue leyendo impresa en gris, porque cada línea trae tres señales además del
color: su estilo de trazo, su marcador y su nombre escrito al final.

## El texto alternativo

Una gráfica en un reporte o en una página necesita una descripción escrita para quien use
lector de pantalla. Se escribe como la frase que dirías en voz alta si la imagen no cargara:
qué muestra, y qué te muestra a ti.

Y se escribe mirando la tabla, no de memoria. Describir una tendencia que los datos no tienen
es la forma más fácil de que una gráfica accesible diga algo falso.

In [ ]:
print((por_canal_mes / 1000).round(0).to_string())

In [ ]:
texto_alt = (
    "Gráfica de líneas del ingreso de 2025 por mes para tres canales de venta, en miles "
    "de pesos. Retail se mantiene entre 120 y 320 todo el año. Online oscila entre 36 y "
    "656 sin tendencia clara. Mayoreo es el canal más grande en diez de los doce meses y "
    "salta de 322 en noviembre a 1,611 en diciembre, que es lo que produce el pico de fin "
    "de año."
)
print(texto_alt)

Cada cifra de ese párrafo se puede verificar contra la tabla de arriba, y por eso se puede
escribir sin miedo. Compruébalo tú.

In [ ]:
tabla = (por_canal_mes / 1000).round(0)

print("Retail va de", tabla["Retail"].min(), "a", tabla["Retail"].max())
print("Online va de", tabla["Online"].min(), "a", tabla["Online"].max())
print("Meses en que Mayoreo es el más grande:",
      (tabla.idxmax(axis=1) == "Wholesale").sum(), "de 12")
print("Mayoreo en noviembre:", tabla["Wholesale"].iloc[10],
      "| en diciembre:", tabla["Wholesale"].iloc[11])

---
## Cuatro formas de arruinar una gráfica correcta

**Cortar el eje vertical.** Una diferencia del dos por ciento se ve como del cincuenta. Los
números están bien y la gráfica miente. Ya lo viste con North contra South.

**Barras sin ordenar.** El lector tiene que hacer el ranking a ojo. Ordenarlas es gratis y
contesta la pregunta sola.

**Línea sobre categorías.** Conectar Norte con Sur sugiere un recorrido que no existe. Para
categorías van barras.

**Dejar el título por omisión.** Una gráfica sin título ni fuente es una opinión. Con las dos
cosas es evidencia.

---
# Ejercicios

Las soluciones están hasta abajo.

### Ejercicio 1 · Elegir sin dibujar

Para cada pregunta, di en un comentario qué gráfica usarías y por qué. No dibujes nada
todavía.

1. ¿Cuál de los tres canales vende más?
2. ¿El ingreso creció o bajó a lo largo del año?
3. ¿Los meses con más ventas son los de mayor ticket promedio?
4. ¿Qué tan parejo es el tamaño de las ventas?

### Ejercicio 2 · Las cuatro, con tus datos

Dibuja una de cada tipo usando las tablas del curso: una barra, una línea, una dispersión y un
histograma. Ponles a todas título, etiqueta de eje y base en cero donde aplique.

Usa una cuadrícula de dos por dos, con `plt.subplots(2, 2)`.

### Ejercicio 3 · De descripción a hallazgo

Toma la gráfica de ingreso por región y escríbele tres títulos distintos:

1. Uno que describa los ejes.
2. Uno que diga el hallazgo con una cifra.
3. Uno que diga el hallazgo con una comparación.

Dibuja la tercera versión completa, con subtítulo, ejes formateados y fuente.

### Ejercicio 4 · El histograma de las ventas

Haz un histograma de la columna `amount` de `ventas`. Dibuja encima el promedio y la mediana,
con colores y estilos distintos, y etiqueta las dos.

Después contesta en un comentario cuál de las dos describe mejor una venta típica, y por qué
están tan separadas.

### Ejercicio 5 · La misma cifra, honesta y tramposa

Toma el ingreso por canal y dibuja dos versiones lado a lado: una con el eje empezando en
cero, y otra con el eje cortado para que la diferencia parezca enorme.

Calcula e imprime la proporción real entre el canal mayor y el menor, para que quede claro
cuánto exagera la segunda.

### Ejercicio 6 · Texto alternativo verificable

Escribe el texto alternativo de la gráfica del ejercicio 3. Después escribe el código que
comprueba cada cifra que mencionaste, como se hizo arriba.

Si alguna cifra no se puede comprobar con una línea de pandas, quítala del texto.

### Ejercicio 7 · Una gráfica de tu proyecto, terminada

Produce una gráfica con los datos de tu proyecto: título que diga el hallazgo, subtítulo
descriptivo, ejes formateados, un elemento resaltado y la fuente al pie. Escribe también su
texto alternativo.

Nada de pastel, y el eje vertical empieza en cero.

La prueba: enséñala sin decir nada. Si tu compañero pregunta qué muestra, al título le falta
el hallazgo.

---
## Tres ideas para llevarse

**La pregunta elige la gráfica.** Barra compara, línea cambia con el tiempo, dispersión
relaciona e histograma reparte. Elegir la forma primero y buscarle datos después es como
salen las gráficas bonitas que no dicen nada.

**Titula con el hallazgo.** El nombre de los ejes ya se ve. Lo que el lector no puede ver solo
es qué encontraste tú.

**El color nunca va solo.** Estilo de línea, marcador o etiqueta directa. Todo eso sobrevive a
una impresión en gris y a quien no distingue dos de tus colores.

La siguiente sesión es seaborn, que hace en una línea varias de las que hoy tomaron ocho, y el
cierre del proyecto integrador.

---
# Soluciones

### Ejercicio 1

```python
# 1. Barras. Son tres categorías con nombre y la pregunta es cómo se comparan.
#    Ordenadas, para que el orden conteste solo.
# 2. Línea. El eje es el tiempo y los meses van en un orden que significa algo.
# 3. Dispersión. Son dos cifras por mes y la pregunta es si se mueven juntas.
#    Un punto por mes, ventas en un eje y ticket promedio en el otro.
# 4. Histograma. Es una sola columna y la pregunta es cómo se reparte, no cómo
#    se compara contra otra cosa.
```

La cuarta es la que más se falla. "Qué tan parejo" suena a comparación y no lo es: hay una
sola variable, y lo que se quiere ver es su forma.

### Ejercicio 2

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

por_canal = ventas.groupby("channel")["amount"].sum().sort_values() / 1000
axes[0, 0].barh(por_canal.index, por_canal.values, color="#2B5F8F")
axes[0, 0].set_title("Mayoreo trae la mitad del ingreso", loc="left", fontweight="bold")
axes[0, 0].set_xlabel("Miles de pesos")

axes[0, 1].plot(MESES, mensual.values / 1000, marker="o", color="#2B5F8F", linewidth=2)
axes[0, 1].set_title("Diciembre rompe la escala", loc="left", fontweight="bold")
axes[0, 1].set_ylabel("Miles de pesos")
axes[0, 1].set_ylim(bottom=0)

axes[1, 0].scatter(ventas["units"], ventas["amount"] / 1000,
                   alpha=0.5, color="#2B5F8F", edgecolor="none")
axes[1, 0].set_title("Más unidades no siempre es más dinero", loc="left", fontweight="bold")
axes[1, 0].set_xlabel("Unidades")
axes[1, 0].set_ylabel("Miles de pesos")

axes[1, 1].hist(ventas["units"], bins=20, color="#2B5F8F", edgecolor="white")
axes[1, 1].set_title("La mayoría de las ventas son chicas", loc="left", fontweight="bold")
axes[1, 1].set_xlabel("Unidades")
axes[1, 1].set_ylabel("Ventas")

for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()
```

La dispersión de abajo a la izquierda es la interesante: se ven bandas horizontales, una por
producto, porque el monto es unidades por un precio que solo toma cinco valores. Una gráfica
puede enseñarte la estructura del archivo además de la respuesta que buscabas.

### Ejercicio 3

```python
# 1. "Ingreso por región"                       describe los ejes
# 2. "North concentró el 34 % del ingreso"      hallazgo con cifra
# 3. "North vende casi el triple que South"     hallazgo con comparación

parte_norte = por_region["North"] / por_region.sum()
veces = por_region["North"] / por_region["South"]

orden = por_region.sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
barras = ax.barh(orden.index, orden.values, color="#C7D6E8")
barras[-1].set_color("#2B5F8F")

ax.set_title(f"North vende {veces:.1f} veces lo de South",
             fontsize=15, fontweight="bold", loc="left", pad=18)
ax.text(0, 1.04, "Ingreso por región, 2025", transform=ax.transAxes,
        fontsize=10.5, color="#5B6B84")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / 1000:.1f}M"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.text(0.125, -0.03, "Fuente: sales_clean.csv, 306 registros, 2025",
         fontsize=9, color="#5B6B84")
plt.show()
```

La tercera es la más útil de las tres porque no obliga al lector a saber si 34 % es mucho.
Una comparación trae su propia referencia.

### Ejercicio 4

```python
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(ventas["amount"] / 1000, bins=30, color="#2B5F8F", edgecolor="white")

promedio = ventas["amount"].mean() / 1000
mediana = ventas["amount"].median() / 1000

ax.axvline(promedio, color="#B4530A", linestyle="--", linewidth=2)
ax.axvline(mediana, color="#0B1B3A", linestyle=":", linewidth=2)
ax.annotate(f"promedio {promedio:,.0f}k", xy=(promedio, 0), xytext=(6, 40),
            textcoords="offset points", color="#B4530A", fontweight="bold")
ax.annotate(f"mediana {mediana:,.0f}k", xy=(mediana, 0), xytext=(-90, 60),
            textcoords="offset points", color="#0B1B3A", fontweight="bold")

ax.set_title("La venta típica es mucho más chica que el promedio",
             loc="left", fontweight="bold")
ax.set_xlabel("Miles de pesos por venta")
ax.set_ylabel("Ventas")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

# La mediana describe mejor una venta típica. El reparto tiene una cola larga a la
# derecha: unas pocas ventas de máquina de espresso valen veinte veces lo que una
# de tarros, y esas jalan el promedio hacia arriba sin que la mayoría se le acerque.
```

Esta es la razón por la que un reporte serio da promedio y mediana juntos. Cuando se separan
tanto, la separación es el hallazgo.

### Ejercicio 5

```python
canal = ventas.groupby("channel")["amount"].sum() / 1000

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(canal.index, canal.values, color="#2B5F8F")
axes[0].set_ylim(bottom=0)
axes[0].set_title("Honesta", loc="left", fontweight="bold")

axes[1].bar(canal.index, canal.values, color="#B4530A")
axes[1].set_ylim(canal.min() * 0.97, canal.max() * 1.01)
axes[1].set_title("Tramposa: el eje empieza cerca del mínimo", loc="left", fontweight="bold")

for ax in axes:
    ax.set_ylabel("Miles de pesos")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

print("Proporción real mayor contra menor:", round(canal.max() / canal.min(), 2))
```

La proporción real es cercana a cuatro y la versión tramposa la hace ver como veinte. Vale la
pena dibujarla una vez, porque es la manipulación que más vas a encontrar en gráficas ajenas.

### Ejercicio 6

```python
texto = (
    "Gráfica de barras horizontales del ingreso de 2025 por región, en millones de "
    "pesos. North es la más alta con 4.35 millones, seguida de Centre con 3.92 y West "
    "con 3.03. South es la más baja con 1.55 millones, casi un tercio de North."
)
print(texto)

print("North:", round(por_region['North'] / 1000, 2), "millones")
print("Centre:", round(por_region['Centre'] / 1000, 2))
print("West:", round(por_region['West'] / 1000, 2))
print("South:", round(por_region['South'] / 1000, 2))
print("South como parte de North:", round(por_region['South'] / por_region['North'], 2))
```

Nota que la descripción da el orden y las cifras, no adjetivos. "North domina claramente" no
le sirve a nadie que no pueda ver la gráfica; "4.35 contra 1.55 millones" sí.

### Ejercicio 7

No hay solución publicada porque los datos son distintos para cada quien. Se califica sobre
cinco cosas: título con hallazgo, subtítulo descriptivo, ejes formateados, un elemento
resaltado y la fuente al pie. El texto alternativo se califica aparte, y cada cifra que
mencione tiene que poder comprobarse.